# Capstone BBQ — Analysis Notebook

Universal experiment runner. Adjust `CONFIG` and re-run to compare surrogates, acquisition functions, and hyperparameters across all selected functions.

In [ ]:
CONFIG = {
    # which functions to run (1–8)
    "functions": [1, 2, 3, 4, 5, 6, 7, 8],

    # which surrogates to compare
    "methods": ["GP", "SVR", "MLP"],

    # hyperparameters per surrogate
    "hyperparams": {
        "GP":  {"n_restarts": 5},
        "SVR": {"C": 1.0, "epsilon": 0.1, "n_bootstrap": 10},
        "MLP": {"hidden": 64, "n_layers": 2, "n_ensemble": 5, "epochs": 500, "lr": 1e-3},
    },

    # acquisition function: 'EI', 'UCB', or 'PI'
    "acquisition": "EI",
    "acquisition_params": {"xi": 0.01},

    # set True if optimising for maximum y, False for minimum
    "maximize": False,

    # candidate grid resolution (per dimension)
    "n_candidates": 1000,

    # how many next points to suggest per function
    "n_suggestions": 3,
}

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# make src importable when running from notebooks/
sys.path.insert(0, str(Path("__file__").resolve().parent.parent))

from src.loader import get_observations, summary
from src.surrogates.gp  import GPSurrogate
from src.surrogates.svr import SVRSurrogate
from src.surrogates.mlp import MLPSurrogate
from src.acquisition import get_acquisition, suggest_next

SURROGATE_REGISTRY = {
    "GP":  GPSurrogate,
    "SVR": SVRSurrogate,
    "MLP": MLPSurrogate,
}

## Data overview

In [ ]:
summary()

## Run surrogates and generate suggestions

In [ ]:
acq = get_acquisition(
    CONFIG["acquisition"],
    maximize=CONFIG["maximize"],
    **CONFIG["acquisition_params"],
)

# results[fn_id][method] = {"mean", "std", "suggestions", "scores"}
results = {}

for fn_id in CONFIG["functions"]:
    X, y = get_observations(fn_id)
    if len(X) == 0:
        print(f"function {fn_id:02d}: no observations, skipping")
        continue

    input_dim = X.shape[1]
    y_best = y.max() if CONFIG["maximize"] else y.min()

    # random candidate grid in [0, 1]^d
    rng = np.random.default_rng(42)
    X_candidates = rng.random((CONFIG["n_candidates"], input_dim))

    results[fn_id] = {}

    for method in CONFIG["methods"]:
        params = CONFIG["hyperparams"].get(method, {})
        model  = SURROGATE_REGISTRY[method](**params)

        try:
            model.fit(X, y)
            mean, std = model.predict(X_candidates)
            X_next, scores = suggest_next(
                X_candidates, mean, std, y_best, acq,
                n=CONFIG["n_suggestions"], exclude=X,
            )
            results[fn_id][method] = {
                "mean": mean, "std": std,
                "suggestions": X_next, "scores": scores,
                "error": None,
            }
            print(f"fn {fn_id:02d} | {method:4s} | y_best={y_best:.4g} | top score={scores[0]:.4g}")
        except Exception as e:
            results[fn_id][method] = {"error": str(e)}
            print(f"fn {fn_id:02d} | {method:4s} | ERROR: {e}")

## Suggestions per function

In [ ]:
for fn_id, fn_results in results.items():
    X, y = get_observations(fn_id)
    print(f"\n{'='*60}")
    print(f"Function {fn_id:02d} | dim={X.shape[1]} | n_obs={len(X)} | y_best={y.min():.6g}")
    print(f"{'='*60}")

    for method, res in fn_results.items():
        if res["error"]:
            print(f"  {method}: ERROR — {res['error']}")
            continue
        print(f"\n  {method} — top {CONFIG['n_suggestions']} suggestions:")
        for i, (x, s) in enumerate(zip(res["suggestions"], res["scores"])):
            x_fmt = ", ".join(f"{v:.4f}" for v in x)
            print(f"    [{i+1}] x=[{x_fmt}]  score={s:.4g}")

## Uncertainty comparison across surrogates

In [ ]:
for fn_id, fn_results in results.items():
    methods_ok = [m for m, r in fn_results.items() if not r["error"]]
    if not methods_ok:
        continue

    fig, axes = plt.subplots(1, len(methods_ok), figsize=(5 * len(methods_ok), 3), sharey=True)
    if len(methods_ok) == 1:
        axes = [axes]

    fig.suptitle(f"Function {fn_id:02d} — predicted std across candidate grid", fontsize=12)

    for ax, method in zip(axes, methods_ok):
        std = fn_results[method]["std"]
        ax.hist(std, bins=30, edgecolor="none")
        ax.set_title(method)
        ax.set_xlabel("std")
        ax.set_ylabel("count")

    plt.tight_layout()
    plt.show()

## Summary table

In [ ]:
rows = []
for fn_id, fn_results in results.items():
    X, y = get_observations(fn_id)
    for method, res in fn_results.items():
        if res["error"]:
            continue
        rows.append({
            "function":    fn_id,
            "dim":         X.shape[1],
            "n_obs":       len(X),
            "method":      method,
            "y_best":      y.min() if not CONFIG["maximize"] else y.max(),
            "top_score":   res["scores"][0],
            "mean_std":    res["std"].mean().round(4),
        })

df = pd.DataFrame(rows)
df